
# Neural Machine Translation: English → Amharic

**Basic Seq2Seq+LSTM vs. Attention-based Seq2Seq+LSTM**

This notebook implements the full pipeline requested for the project:

1. Dataset & Preprocessing
2. Model Development & Training (2 required models)
3. Model Evaluation & Comparison (BLEU, chrF, timing, params)
4. Error & Attention Analysis
5. Deployment (Streamlit app, generated as a standalone `app.py`)

> **Note on scale:** This dataset (~17.5k sentence pairs, King James-style English aligned with
> Amharic/Ge'ez, likely a digitized Bible corpus) is small by NMT standards, and word-level LSTM
> seq2seq models are data-hungry. On CPU, full training (say 15-30 epochs at the hyperparameters
> below) can take a long time. Recommendations if you need it to run faster:
> - Set `QUICK_MODE = True` below to train on a subset with fewer epochs (for pipeline testing).
> - Otherwise, run this notebook on a GPU (Colab/Kaggle) — just install `torch` with CUDA there.
> - Everything below (architectures, training loop, evaluation, error analysis, deployment) is unchanged either way.

---

## 0. Dataset documentation (fill in for your report)

Fill these in based on where you actually obtained the CSV (`amh, gez, eng` columns), since exact
source/license should be documented from your download, not guessed:

- **Source:** _e.g. name of the digital Bible corpus / repository / Hugging Face dataset you downloaded this from_
- **License:** _check the source's license page — many digitized scripture corpora are public domain or CC-licensed, but confirm before submitting_
- **Size:** 17,516 aligned sentence triples (English / Amharic / Ge'ez)
- **Characteristics:** Long-form, archaic/formal register (King James Version-style English), religious/narrative domain, average English sentence ≈25 words (max 80), average Amharic sentence ≈14 words (max 53) — this length mismatch matters for tokenization/padding choices below.
- **Direction used in this notebook:** English → Amharic (the `gez` Ge'ez column is loaded but unused; drop it or extend to a 3-way system if your group wants to compare Ge'ez as well).


## 1. Setup & Imports

In [ ]:
# If running fresh (e.g. in Colab), uncomment:
!pip install torch sacrebleu nltk matplotlib pandas -q

import os
import re
import time
import math
import random
import pickle
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Toggle this ON for a fast smoke-test run of the whole pipeline (small subset, few epochs).
# Toggle OFF for the real training run for your report numbers.
QUICK_MODE = True

DATA_PATH = "AGE.csv"
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Dataset & Preprocessing

### 2.1 Load and inspect

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(df_raw.shape)
df_raw.head()

In [ ]:
# Keep only the columns we need for English -> Amharic
df = df_raw[["eng", "amh"]].copy()

print("Rows:", len(df))
print("Missing values:\n", df.isna().sum())
print("Exact duplicate pairs:", df.duplicated().sum())

### 2.2 Cleaning & normalization

- Strip whitespace, normalize repeated spaces
- Drop rows with missing source or target
- Drop exact duplicate (source, target) pairs
- Normalize quote characters and remove stray control characters
- Filter out empty / whitespace-only strings and pairs with extreme length ratios (likely misalignment) or that exceed a max length

In [ ]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)                  # collapse whitespace
    text = text.replace("\u200b", "")                  # zero-width space sometimes present in Ethiopic text
    text = text.replace("\u201c", '"').replace("\u201d", '"')   # smart double quotes -> straight
    text = text.replace("\u2018", "'").replace("\u2019", "'")   # smart single quotes -> straight
    return text.strip()

df["eng"] = df["eng"].apply(normalize_text)
df["amh"] = df["amh"].apply(normalize_text)

# Drop missing/empty
before = len(df)
df = df[(df["eng"].str.len() > 0) & (df["amh"].str.len() > 0)]
print(f"Dropped {before - len(df)} rows with empty source/target")

# Drop exact duplicates
before = len(df)
df = df.drop_duplicates(subset=["eng", "amh"])
print(f"Dropped {before - len(df)} exact duplicate pairs")

df = df.reset_index(drop=True)
print("Rows after cleaning:", len(df))

### 2.3 Tokenization

We use simple, dependency-free regex word tokenizers (adequate for building a from-scratch
vocabulary/LSTM pipeline). Amharic punctuation (`፡ ። ፣ ፤ ፥ ፦ ፧`) is treated as separate tokens,
same as English punctuation.

**Note:** if your group wants stronger handling of Amharic morphology (a morphologically rich
language), consider swapping this for a subword tokenizer (e.g. SentencePiece/BPE trained on the
Amharic side) — that would help a lot with rare/unknown words in the error analysis in Part 4.
This word-level version is kept here to match the "vocabulary/tokenization mechanisms" requirement
in the simplest, most transparent way.


In [ ]:
AMH_PUNCT = "፡።፣፤፥፦፧"
_en_token_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[.,!?;:\"()\-]")
_am_token_re = re.compile(rf"[^\s{AMH_PUNCT}]+|[{AMH_PUNCT}]")

def tokenize_en(text):
    return _en_token_re.findall(text.lower())

def tokenize_am(text):
    return _am_token_re.findall(text)

print(tokenize_en(df.loc[0, "eng"]))
print(tokenize_am(df.loc[0, "amh"]))

In [ ]:
MAX_LEN = 60      # tokens, applied to both sides after tokenization
MIN_LEN = 1

df["eng_toks"] = df["eng"].apply(tokenize_en)
df["amh_toks"] = df["amh"].apply(tokenize_am)

df["eng_len"] = df["eng_toks"].apply(len)
df["amh_len"] = df["amh_toks"].apply(len)

before = len(df)
df = df[(df["eng_len"].between(MIN_LEN, MAX_LEN)) & (df["amh_len"].between(MIN_LEN, MAX_LEN))]
print(f"Dropped {before - len(df)} rows outside length bounds [{MIN_LEN}, {MAX_LEN}]")

# Filter out likely misaligned pairs via extreme length ratio (heuristic sanity filter)
ratio = df["eng_len"] / df["amh_len"].clip(lower=1)
before = len(df)
df = df[(ratio > 0.15) & (ratio < 8)]
print(f"Dropped {before - len(df)} rows with extreme length ratio (likely misalignment)")

df = df.reset_index(drop=True)
print("Final cleaned rows:", len(df))
df[["eng_len", "amh_len"]].describe()

### 2.4 Vocabulary

In [ ]:
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = "<pad>", "<sos>", "<eos>", "<unk>"

class Vocab:
    def __init__(self, token_lists, min_freq=2):
        counter = Counter()
        for toks in token_lists:
            counter.update(toks)
        self.itos = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
        for tok, freq in counter.most_common():
            if freq >= min_freq:
                self.itos.append(tok)
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def encode(self, tokens):
        unk = self.stoi[UNK_TOKEN]
        ids = [self.stoi[SOS_TOKEN]] + [self.stoi.get(t, unk) for t in tokens] + [self.stoi[EOS_TOKEN]]
        return ids

    def decode(self, ids, strip_special=True):
        toks = [self.itos[i] for i in ids]
        if strip_special:
            toks = [t for t in toks if t not in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN)]
        return toks

    def __len__(self):
        return len(self.itos)


MIN_FREQ = 2

### 2.5 Train / validation / test split (80 / 10 / 10)

In [ ]:
from sklearn.model_selection import train_test_split

if QUICK_MODE:
    df = df.sample(n=min(3000, len(df)), random_state=SEED).reset_index(drop=True)
    print("QUICK_MODE: subsampled to", len(df), "rows")

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# Build vocabs from TRAINING split only (avoid test leakage)
src_vocab = Vocab(train_df["eng_toks"].tolist(), min_freq=MIN_FREQ)
trg_vocab = Vocab(train_df["amh_toks"].tolist(), min_freq=MIN_FREQ)
print("English vocab size:", len(src_vocab))
print("Amharic vocab size:", len(trg_vocab))

### 2.6 PyTorch Dataset & DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, dataframe, src_vocab, trg_vocab):
        self.src_ids = [torch.tensor(src_vocab.encode(t), dtype=torch.long)
                        for t in dataframe["eng_toks"]]
        self.trg_ids = [torch.tensor(trg_vocab.encode(t), dtype=torch.long)
                         for t in dataframe["amh_toks"]]
        self.src_text = dataframe["eng"].tolist()
        self.trg_text = dataframe["amh"].tolist()

    def __len__(self):
        return len(self.src_ids)

    def __getitem__(self, idx):
        return self.src_ids[idx], self.trg_ids[idx]


def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_lens = torch.tensor([len(s) for s in src_batch])
    src_padded = pad_sequence(src_batch, padding_value=src_vocab.stoi[PAD_TOKEN])
    trg_padded = pad_sequence(trg_batch, padding_value=trg_vocab.stoi[PAD_TOKEN])
    return src_padded, src_lens, trg_padded   # shape: [seq_len, batch]


train_ds = TranslationDataset(train_df, src_vocab, trg_vocab)
val_ds = TranslationDataset(val_df, src_vocab, trg_vocab)
test_ds = TranslationDataset(test_df, src_vocab, trg_vocab)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

xb, xl, yb = next(iter(train_loader))
print("src batch:", xb.shape, "trg batch:", yb.shape)

## 3. Model 1 — Basic Seq2Seq + LSTM

Standard encoder-decoder: the encoder LSTM compresses the source sentence into a final
hidden/cell state; the decoder LSTM is initialized from that state and generates the target
sentence one token at a time (no attention — this is the required baseline).


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=src_vocab.stoi[PAD_TOKEN])
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout if n_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [src_len, batch]
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=trg_vocab.stoi[PAD_TOKEN])
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout if n_layers > 1 else 0)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_tok, hidden, cell):
        # input_tok: [batch]  (single time step)
        input_tok = input_tok.unsqueeze(0)              # [1, batch]
        embedded = self.dropout(self.embedding(input_tok))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))      # [batch, output_dim]
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [src_len, batch], trg: [trg_len, batch]
        trg_len, batch_size = trg.shape
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        input_tok = trg[0, :]   # <sos>
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_tok, hidden, cell)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_tok = trg[t] if teacher_force else top1
        return outputs

    @torch.no_grad()
    def translate(self, src, src_vocab, trg_vocab, max_len=60):
        self.eval()
        hidden, cell = self.encoder(src)
        input_tok = torch.tensor([trg_vocab.stoi[SOS_TOKEN]], device=self.device)
        result_ids = []
        for _ in range(max_len):
            output, hidden, cell = self.decoder(input_tok, hidden, cell)
            top1 = output.argmax(1)
            token_id = top1.item()
            if token_id == trg_vocab.stoi[EOS_TOKEN]:
                break
            result_ids.append(token_id)
            input_tok = top1
        return trg_vocab.decode(result_ids, strip_special=True)

### 3.1 Hyperparameters & training configuration (Model 1)

In [ ]:
CONFIG_1 = dict(
    EMB_DIM=256,
    HID_DIM=512,
    N_LAYERS=2,
    DROPOUT=0.5,
    BATCH_SIZE=BATCH_SIZE,
    LEARNING_RATE=1e-3,
    OPTIMIZER="Adam",
    N_EPOCHS=3 if QUICK_MODE else 20,
    CLIP=1.0,
    TEACHER_FORCING_RATIO=0.5,
    LOSS_FN="CrossEntropyLoss (ignore_index=<pad>)",
)
CONFIG_1

In [ ]:
enc1 = Encoder(len(src_vocab), CONFIG_1["EMB_DIM"], CONFIG_1["HID_DIM"], CONFIG_1["N_LAYERS"], CONFIG_1["DROPOUT"])
dec1 = Decoder(len(trg_vocab), CONFIG_1["EMB_DIM"], CONFIG_1["HID_DIM"], CONFIG_1["N_LAYERS"], CONFIG_1["DROPOUT"])
model1 = Seq2Seq(enc1, dec1, DEVICE).to(DEVICE)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model 1 (basic Seq2Seq) trainable parameters: {count_parameters(model1):,}")

optimizer1 = optim.Adam(model1.parameters(), lr=CONFIG_1["LEARNING_RATE"])
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab.stoi[PAD_TOKEN])

### 3.2 Generic training / evaluation loop (reused for both models)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip, teacher_forcing_ratio=0.5):
    model.train()
    epoch_loss = 0
    for src, src_lens, trg in loader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim)
        trg_flat = trg[1:].reshape(-1)
        loss = criterion(output, trg_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)


@torch.no_grad()
def evaluate_epoch(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    for src, src_lens, trg in loader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        output = model(src, trg, teacher_forcing_ratio=0.0)   # no teacher forcing at eval
        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim)
        trg_flat = trg[1:].reshape(-1)
        loss = criterion(output, trg_flat)
        epoch_loss += loss.item()
    return epoch_loss / len(loader)


def run_training(model, optimizer, criterion, config, model_name, checkpoint_path):
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}
    start_time = time.time()
    for epoch in range(config["N_EPOCHS"]):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion,
                                  config["CLIP"], config["TEACHER_FORCING_RATIO"])
        val_loss = evaluate_epoch(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
        print(f"[{model_name}] Epoch {epoch+1}/{config['N_EPOCHS']} "
              f"| train_loss={train_loss:.3f} | val_loss={val_loss:.3f} "
              f"| PPL={math.exp(val_loss):.2f} | time={time.time()-t0:.1f}s")
    total_time = time.time() - start_time
    print(f"[{model_name}] Total training time: {total_time:.1f}s")
    return history, total_time

### 3.3 Train Model 1

In [ ]:
history1, train_time1 = run_training(
    model1, optimizer1, criterion, CONFIG_1,
    "Seq2Seq-LSTM", os.path.join(CHECKPOINT_DIR, "model1_seq2seq.pt")
)
model1.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "model1_seq2seq.pt")))

## 4. Model 2 — Attention-based Seq2Seq + LSTM

Bahdanau-style additive attention: a bidirectional LSTM encoder produces per-timestep outputs;
at every decoding step, the decoder computes attention weights over **all** encoder outputs
(instead of relying on a single fixed context vector as in Model 1), builds a weighted context
vector, and uses it alongside the current input token to predict the next word. This should
help especially with longer sentences and word alignment.


In [ ]:
class AttnEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=src_vocab.stoi[PAD_TOKEN])
        self.rnn = nn.LSTM(emb_dim, enc_hid_dim, n_layers, bidirectional=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc_hidden = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.fc_cell = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.n_layers = n_layers

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs: [src_len, batch, enc_hid_dim*2]  (per-timestep, for attention)
        # hidden/cell: [n_layers*2, batch, enc_hid_dim] -> combine fwd/bwd per layer for decoder init
        def combine(state, fc):
            state = state.view(self.n_layers, 2, state.shape[1], -1)   # [layers, dirs, batch, hid]
            fwd, bwd = state[:, 0], state[:, 1]
            combined = torch.cat((fwd, bwd), dim=2)                    # [layers, batch, hid*2]
            return torch.tanh(fc(combined))
        hidden_dec = combine(hidden, self.fc_hidden)
        cell_dec = combine(cell, self.fc_cell)
        return outputs, hidden_dec, cell_dec


class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden_top, encoder_outputs):
        # hidden_top: [batch, dec_hid_dim] (top decoder layer's hidden state)
        # encoder_outputs: [src_len, batch, enc_hid_dim*2]
        src_len = encoder_outputs.shape[0]
        hidden_rep = hidden_top.unsqueeze(1).repeat(1, src_len, 1)          # [batch, src_len, dec_hid]
        encoder_outputs = encoder_outputs.permute(1, 0, 2)                  # [batch, src_len, enc_hid*2]
        energy = torch.tanh(self.attn(torch.cat((hidden_rep, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)                               # [batch, src_len]
        return torch.softmax(attention, dim=1)


class AttnDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=trg_vocab.stoi[PAD_TOKEN])
        self.rnn = nn.LSTM(emb_dim + enc_hid_dim * 2, dec_hid_dim, n_layers,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc_out = nn.Linear(emb_dim + enc_hid_dim * 2 + dec_hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_tok, hidden, cell, encoder_outputs):
        input_tok = input_tok.unsqueeze(0)                       # [1, batch]
        embedded = self.dropout(self.embedding(input_tok))       # [1, batch, emb_dim]

        attn_weights = self.attention(hidden[-1], encoder_outputs)          # [batch, src_len]
        attn_weights_u = attn_weights.unsqueeze(1)                          # [batch, 1, src_len]
        enc_out_b = encoder_outputs.permute(1, 0, 2)                        # [batch, src_len, enc_hid*2]
        context = torch.bmm(attn_weights_u, enc_out_b)                      # [batch, 1, enc_hid*2]
        context = context.permute(1, 0, 2)                                  # [1, batch, enc_hid*2]

        rnn_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))

        embedded = embedded.squeeze(0)
        output = output.squeeze(0)
        context = context.squeeze(0)
        prediction = self.fc_out(torch.cat((output, context, embedded), dim=1))
        return prediction, hidden, cell, attn_weights


class AttnSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len, batch_size = trg.shape
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)
        input_tok = trg[0, :]
        for t in range(1, trg_len):
            output, hidden, cell, _ = self.decoder(input_tok, hidden, cell, encoder_outputs)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_tok = trg[t] if teacher_force else top1
        return outputs

    @torch.no_grad()
    def translate(self, src, src_vocab, trg_vocab, max_len=60, return_attention=False):
        self.eval()
        encoder_outputs, hidden, cell = self.encoder(src)
        input_tok = torch.tensor([trg_vocab.stoi[SOS_TOKEN]], device=self.device)
        result_ids, attn_history = [], []
        for _ in range(max_len):
            output, hidden, cell, attn_weights = self.decoder(input_tok, hidden, cell, encoder_outputs)
            top1 = output.argmax(1)
            token_id = top1.item()
            attn_history.append(attn_weights.squeeze(0).cpu().numpy())
            if token_id == trg_vocab.stoi[EOS_TOKEN]:
                break
            result_ids.append(token_id)
            input_tok = top1
        words = trg_vocab.decode(result_ids, strip_special=True)
        if return_attention:
            return words, np.array(attn_history[:len(words)] if words else attn_history)
        return words

### 4.1 Hyperparameters & training configuration (Model 2)

In [ ]:
CONFIG_2 = dict(
    EMB_DIM=256,
    ENC_HID_DIM=512,
    DEC_HID_DIM=512,
    N_LAYERS=2,
    DROPOUT=0.5,
    BATCH_SIZE=BATCH_SIZE,
    LEARNING_RATE=1e-3,
    OPTIMIZER="Adam",
    N_EPOCHS=3 if QUICK_MODE else 20,
    CLIP=1.0,
    TEACHER_FORCING_RATIO=0.5,
    LOSS_FN="CrossEntropyLoss (ignore_index=<pad>)",
)
CONFIG_2

In [ ]:
attn = Attention(CONFIG_2["ENC_HID_DIM"], CONFIG_2["DEC_HID_DIM"])
enc2 = AttnEncoder(len(src_vocab), CONFIG_2["EMB_DIM"], CONFIG_2["ENC_HID_DIM"],
                    CONFIG_2["DEC_HID_DIM"], CONFIG_2["N_LAYERS"], CONFIG_2["DROPOUT"])
dec2 = AttnDecoder(len(trg_vocab), CONFIG_2["EMB_DIM"], CONFIG_2["ENC_HID_DIM"],
                    CONFIG_2["DEC_HID_DIM"], CONFIG_2["N_LAYERS"], CONFIG_2["DROPOUT"], attn)
model2 = AttnSeq2Seq(enc2, dec2, DEVICE).to(DEVICE)

print(f"Model 2 (Attention Seq2Seq) trainable parameters: {count_parameters(model2):,}")

optimizer2 = optim.Adam(model2.parameters(), lr=CONFIG_2["LEARNING_RATE"])

### 4.2 Train Model 2

In [ ]:
history2, train_time2 = run_training(
    model2, optimizer2, criterion, CONFIG_2,
    "Attention-Seq2Seq-LSTM", os.path.join(CHECKPOINT_DIR, "model2_attention.pt")
)
model2.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "model2_attention.pt")))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history1["train_loss"], label="Model 1 train")
plt.plot(history1["val_loss"], label="Model 1 val")
plt.plot(history2["train_loss"], label="Model 2 train", linestyle="--")
plt.plot(history2["val_loss"], label="Model 2 val", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training curves: Basic Seq2Seq vs Attention Seq2Seq")
plt.legend()
plt.show()

## 5. Model Evaluation & Comparison

We report, for both models, on the held-out **test set**:
- BLEU and chrF (via `sacrebleu`, computed on detokenized text)
- Test loss / perplexity
- Training time (captured above) and per-sentence inference time
- Parameter count


In [ ]:
!pip install sacrebleu -q
import sacrebleu

def detok(tokens):
    return " ".join(tokens)

@torch.no_grad()
def generate_translations(model, dataframe, src_vocab, trg_vocab, is_attention, max_len=60, limit=None):
    model.eval()
    hyps, refs, srcs = [], [], []
    rows = dataframe if limit is None else dataframe.iloc[:limit]
    for _, row in rows.iterrows():
        src_ids = torch.tensor(src_vocab.encode(row["eng_toks"]), dtype=torch.long).unsqueeze(1).to(DEVICE)
        if is_attention:
            pred_tokens = model.translate(src_ids, src_vocab, trg_vocab, max_len=max_len)
        else:
            pred_tokens = model.translate(src_ids, src_vocab, trg_vocab, max_len=max_len)
        hyps.append(detok(pred_tokens))
        refs.append(row["amh"])
        srcs.append(row["eng"])
    return srcs, refs, hyps


def measure_inference_time(model, dataframe, src_vocab, trg_vocab, is_attention, n=50):
    n = min(n, len(dataframe))
    sample = dataframe.iloc[:n]
    torch.cuda.synchronize() if DEVICE.type == "cuda" else None
    t0 = time.time()
    _ = generate_translations(model, sample, src_vocab, trg_vocab, is_attention)
    torch.cuda.synchronize() if DEVICE.type == "cuda" else None
    elapsed = time.time() - t0
    return elapsed / n   # seconds per sentence


def bleu_chrf(refs, hyps):
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return bleu.score, chrf.score

In [ ]:
EVAL_LIMIT = 200 if QUICK_MODE else None   # cap test-set size for speed in quick mode

# --- Model 1 ---
src1, refs1, hyps1 = generate_translations(model1, test_df, src_vocab, trg_vocab, is_attention=False, limit=EVAL_LIMIT)
bleu1, chrf1 = bleu_chrf(refs1, hyps1)
test_loss1 = evaluate_epoch(model1, test_loader, criterion)
inf_time1 = measure_inference_time(model1, test_df, src_vocab, trg_vocab, is_attention=False)
params1 = count_parameters(model1)

# --- Model 2 ---
src2, refs2, hyps2 = generate_translations(model2, test_df, src_vocab, trg_vocab, is_attention=True, limit=EVAL_LIMIT)
bleu2, chrf2 = bleu_chrf(refs2, hyps2)
test_loss2 = evaluate_epoch(model2, test_loader, criterion)
inf_time2 = measure_inference_time(model2, test_df, src_vocab, trg_vocab, is_attention=True)
params2 = count_parameters(model2)

comparison = pd.DataFrame([
    {
        "Model": "Basic Seq2Seq+LSTM",
        "BLEU": round(bleu1, 2),
        "chrF": round(chrf1, 2),
        "Test Loss": round(test_loss1, 3),
        "Test PPL": round(math.exp(test_loss1), 2),
        "Training Time (s)": round(train_time1, 1),
        "Inference Time (s/sentence)": round(inf_time1, 4),
        "Parameters": f"{params1:,}",
    },
    {
        "Model": "Attention Seq2Seq+LSTM",
        "BLEU": round(bleu2, 2),
        "chrF": round(chrf2, 2),
        "Test Loss": round(test_loss2, 3),
        "Test PPL": round(math.exp(test_loss2), 2),
        "Training Time (s)": round(train_time2, 1),
        "Inference Time (s/sentence)": round(inf_time2, 4),
        "Parameters": f"{params2:,}",
    },
])
comparison

In [ ]:
better = "Attention Seq2Seq+LSTM" if bleu2 >= bleu1 else "Basic Seq2Seq+LSTM"
print(f"Higher BLEU on the test set: {better}")
print("(Note: with QUICK_MODE=True / few epochs / a small subset, BLEU scores from both models "
      "will be low and mainly useful for sanity-checking the pipeline, not for the report — "
      "re-run with QUICK_MODE=False and more epochs/data for real numbers.)")

### 5.1 Qualitative side-by-side comparison

In [ ]:
N_EXAMPLES = 10
sample_idx = random.sample(range(len(src1)), min(N_EXAMPLES, len(src1)))

qual_df = pd.DataFrame({
    "Source (English)": [src1[i] for i in sample_idx],
    "Reference (Amharic)": [refs1[i] for i in sample_idx],
    "Seq2Seq Output": [hyps1[i] for i in sample_idx],
    "Attention-LSTM Output": [hyps2[i] for i in sample_idx],
})
pd.set_option("display.max_colwidth", None)
qual_df

## 6. Error & Attention Analysis

We look for the common LSTM/NMT failure modes: incorrect word order, missing/extra words,
repeated words (a classic RNN decoding failure), unknown/rare-word handling, and degradation on
long sentences. Then we visualize attention for a few examples from Model 2.


In [ ]:
def analyze_errors(srcs, refs, hyps, src_vocab, label):
    rows = []
    for s, r, h in zip(srcs, refs, hyps):
        h_toks = h.split()
        r_toks = tokenize_am(r)
        len_diff = len(h_toks) - len(r_toks)
        repeated = sum(1 for i in range(1, len(h_toks)) if h_toks[i] == h_toks[i - 1])
        s_toks = tokenize_en(s)
        unk_count = sum(1 for t in s_toks if t not in src_vocab.stoi)
        rows.append({
            "src_len": len(s_toks),
            "ref_len": len(r_toks),
            "hyp_len": len(h_toks),
            "len_diff": len_diff,                 # + = model added words, - = missing words
            "repeated_word_count": repeated,
            "unk_src_tokens": unk_count,
            "empty_output": len(h_toks) == 0,
        })
    edf = pd.DataFrame(rows)
    edf["model"] = label
    return edf

err1 = analyze_errors(src1, refs1, hyps1, src_vocab, "Basic Seq2Seq+LSTM")
err2 = analyze_errors(src2, refs2, hyps2, src_vocab, "Attention Seq2Seq+LSTM")
err_all = pd.concat([err1, err2], ignore_index=True)

summary = err_all.groupby("model").agg(
    avg_len_diff=("len_diff", "mean"),
    pct_with_repeats=("repeated_word_count", lambda x: (x > 0).mean() * 100),
    avg_repeats=("repeated_word_count", "mean"),
    avg_unk_src_tokens=("unk_src_tokens", "mean"),
    pct_empty_output=("empty_output", "mean"),
).round(2)
summary

In [ ]:
# Performance by source-sentence length bucket (long-sentence degradation check)
def bucket_bleu(srcs, refs, hyps, label):
    buckets = [(0, 10), (11, 20), (21, 35), (36, 200)]
    rows = []
    lens = [len(tokenize_en(s)) for s in srcs]
    for lo, hi in buckets:
        idx = [i for i, l in enumerate(lens) if lo <= l <= hi]
        if len(idx) < 3:
            continue
        b_refs = [refs[i] for i in idx]
        b_hyps = [hyps[i] for i in idx]
        score, _ = bleu_chrf(b_refs, b_hyps)
        rows.append({"model": label, "src_len_bucket": f"{lo}-{hi}", "n": len(idx), "BLEU": round(score, 2)})
    return rows

bucket_rows = bucket_bleu(src1, refs1, hyps1, "Basic Seq2Seq+LSTM") + \
              bucket_bleu(src2, refs2, hyps2, "Attention Seq2Seq+LSTM")
pd.DataFrame(bucket_rows).sort_values(["src_len_bucket", "model"])

### 6.1 Manual error inspection

Print a handful of examples side by side and manually tag the error type(s) you observe
(word order, missing/extra words, repetition, morphology, named entities, unknown words,
long-sentence breakdown). Use `err_all` above to pull specific rows worth discussing — e.g.
sort by `repeated_word_count` or `len_diff` to surface the worst cases automatically.


In [ ]:
# Worst repetition cases for Model 1 (baseline), for manual inspection in your report
worst_idx = err1.sort_values("repeated_word_count", ascending=False).head(5).index
for i in worst_idx:
    print("SRC :", src1[i])
    print("REF :", refs1[i])
    print("HYP1:", hyps1[i])
    print("HYP2:", hyps2[i])
    print("-" * 60)

### 6.2 Attention visualization

In [ ]:
def plot_attention(src_sentence, model, src_vocab, trg_vocab, max_len=40):
    src_tokens = tokenize_en(src_sentence)
    src_ids = torch.tensor(src_vocab.encode(src_tokens), dtype=torch.long).unsqueeze(1).to(DEVICE)
    pred_tokens, attn_matrix = model.translate(src_ids, src_vocab, trg_vocab,
                                                max_len=max_len, return_attention=True)
    # attn_matrix: [pred_len, src_len_with_sos_eos] -> full src incl. <sos>/<eos>
    full_src_tokens = [SOS_TOKEN] + src_tokens + [EOS_TOKEN]

    fig, ax = plt.subplots(figsize=(max(6, len(full_src_tokens) * 0.5),
                                     max(4, len(pred_tokens) * 0.5)))
    im = ax.imshow(attn_matrix[:len(pred_tokens)], cmap="viridis")
    ax.set_xticks(range(len(full_src_tokens)))
    ax.set_xticklabels(full_src_tokens, rotation=90)
    ax.set_yticks(range(len(pred_tokens)))
    ax.set_yticklabels(pred_tokens)
    ax.set_xlabel("Source tokens")
    ax.set_ylabel("Generated Amharic tokens")
    fig.colorbar(im, ax=ax)
    plt.title("Attention weights")
    plt.tight_layout()
    plt.show()
    return pred_tokens


for example_src in test_df["eng"].sample(min(3, len(test_df)), random_state=SEED).tolist():
    print("Source:", example_src)
    plot_attention(example_src, model2, src_vocab, trg_vocab)

## 7. Deployment

We deploy the **Attention Seq2Seq+LSTM** model (the stronger of the two) behind a Streamlit UI.
First we save everything the app needs: model weights + vocabularies + the hyperparameters used
to reconstruct the model architecture.


In [ ]:
ARTIFACT_DIR = "deployment_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

torch.save(model2.state_dict(), os.path.join(ARTIFACT_DIR, "attention_model.pt"))

with open(os.path.join(ARTIFACT_DIR, "src_vocab.pkl"), "wb") as f:
    pickle.dump(src_vocab, f)
with open(os.path.join(ARTIFACT_DIR, "trg_vocab.pkl"), "wb") as f:
    pickle.dump(trg_vocab, f)
with open(os.path.join(ARTIFACT_DIR, "config.pkl"), "wb") as f:
    pickle.dump(CONFIG_2, f)

print("Saved deployment artifacts to:", ARTIFACT_DIR)
print(os.listdir(ARTIFACT_DIR))

### 7.1 Standalone inference module

This mirrors the preprocessing/model classes above so the Streamlit app can load everything independently of this notebook's runtime.

In [ ]:
%%writefile inference.py
"""
Standalone inference module for the English -> Amharic Attention Seq2Seq+LSTM model.
Used by both app.py (Streamlit) and any API wrapper (e.g. FastAPI/Flask) you add.
"""
import re
import pickle
import torch
import torch.nn as nn

PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = "<pad>", "<sos>", "<eos>", "<unk>"
_en_token_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[.,!?;:\"()\-]")


def tokenize_en(text):
    return _en_token_re.findall(text.lower())


class Vocab:
    """Re-declared here (rather than imported from the notebook) so this file is standalone."""
    def __init__(self):
        self.itos = []
        self.stoi = {}

    def encode(self, tokens):
        unk = self.stoi[UNK_TOKEN]
        return [self.stoi[SOS_TOKEN]] + [self.stoi.get(t, unk) for t in tokens] + [self.stoi[EOS_TOKEN]]

    def decode(self, ids, strip_special=True):
        toks = [self.itos[i] for i in ids]
        if strip_special:
            toks = [t for t in toks if t not in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN)]
        return toks

    def __len__(self):
        return len(self.itos)


class AttnEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, n_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.LSTM(emb_dim, enc_hid_dim, n_layers, bidirectional=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc_hidden = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.fc_cell = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.n_layers = n_layers

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)

        def combine(state, fc):
            state = state.view(self.n_layers, 2, state.shape[1], -1)
            fwd, bwd = state[:, 0], state[:, 1]
            combined = torch.cat((fwd, bwd), dim=2)
            return torch.tanh(fc(combined))
        return outputs, combine(hidden, self.fc_hidden), combine(cell, self.fc_cell)


class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden_top, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        hidden_rep = hidden_top.unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.attn(torch.cat((hidden_rep, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)


class AttnDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, n_layers, dropout, attention, pad_idx):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.LSTM(emb_dim + enc_hid_dim * 2, dec_hid_dim, n_layers,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc_out = nn.Linear(emb_dim + enc_hid_dim * 2 + dec_hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_tok, hidden, cell, encoder_outputs):
        input_tok = input_tok.unsqueeze(0)
        embedded = self.dropout(self.embedding(input_tok))
        attn_weights = self.attention(hidden[-1], encoder_outputs)
        attn_weights_u = attn_weights.unsqueeze(1)
        enc_out_b = encoder_outputs.permute(1, 0, 2)
        context = torch.bmm(attn_weights_u, enc_out_b).permute(1, 0, 2)
        rnn_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        embedded, output, context = embedded.squeeze(0), output.squeeze(0), context.squeeze(0)
        prediction = self.fc_out(torch.cat((output, context, embedded), dim=1))
        return prediction, hidden, cell, attn_weights


class AttnSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder, self.decoder, self.device = encoder, decoder, device

    @torch.no_grad()
    def translate(self, src, trg_vocab, max_len=60):
        self.eval()
        encoder_outputs, hidden, cell = self.encoder(src)
        input_tok = torch.tensor([trg_vocab.stoi[SOS_TOKEN]], device=self.device)
        result_ids = []
        for _ in range(max_len):
            output, hidden, cell, _ = self.decoder(input_tok, hidden, cell, encoder_outputs)
            top1 = output.argmax(1)
            token_id = top1.item()
            if token_id == trg_vocab.stoi[EOS_TOKEN]:
                break
            result_ids.append(token_id)
            input_tok = top1
        return trg_vocab.decode(result_ids, strip_special=True)


class Translator:
    """Loads all artifacts once and exposes a simple translate(text) -> str method."""

    def __init__(self, artifact_dir="deployment_artifacts", device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        with open(f"{artifact_dir}/src_vocab.pkl", "rb") as f:
            self.src_vocab = pickle.load(f)
        with open(f"{artifact_dir}/trg_vocab.pkl", "rb") as f:
            self.trg_vocab = pickle.load(f)
        with open(f"{artifact_dir}/config.pkl", "rb") as f:
            cfg = pickle.load(f)

        pad_idx_src = self.src_vocab.stoi[PAD_TOKEN]
        pad_idx_trg = self.trg_vocab.stoi[PAD_TOKEN]

        attn = Attention(cfg["ENC_HID_DIM"], cfg["DEC_HID_DIM"])
        enc = AttnEncoder(len(self.src_vocab), cfg["EMB_DIM"], cfg["ENC_HID_DIM"],
                           cfg["DEC_HID_DIM"], cfg["N_LAYERS"], cfg["DROPOUT"], pad_idx_src)
        dec = AttnDecoder(len(self.trg_vocab), cfg["EMB_DIM"], cfg["ENC_HID_DIM"],
                           cfg["DEC_HID_DIM"], cfg["N_LAYERS"], cfg["DROPOUT"], attn, pad_idx_trg)
        self.model = AttnSeq2Seq(enc, dec, self.device).to(self.device)
        self.model.load_state_dict(torch.load(f"{artifact_dir}/attention_model.pt", map_location=self.device))
        self.model.eval()

    def translate(self, text: str, max_len: int = 60) -> str:
        tokens = tokenize_en(text)
        src_ids = torch.tensor(self.src_vocab.encode(tokens), dtype=torch.long).unsqueeze(1).to(self.device)
        pred_tokens = self.model.translate(src_ids, self.trg_vocab, max_len=max_len)
        return " ".join(pred_tokens)

### 7.2 Streamlit app (`app.py`)

Run with: `streamlit run app.py` (from the directory containing `inference.py` and the `deployment_artifacts/` folder produced above).

In [ ]:
%%writefile app.py
import streamlit as st
from inference import Translator

st.set_page_config(page_title="English -> Amharic Translator", page_icon="🌍")
st.title("English → Amharic Neural Machine Translation")
st.caption("Attention-based Seq2Seq + LSTM")

@st.cache_resource
def load_translator():
    return Translator(artifact_dir="deployment_artifacts")

translator = load_translator()

text = st.text_area("Enter an English sentence:", "I am going to the university.")

if st.button("Translate") and text.strip():
    with st.spinner("Translating..."):
        translation = translator.translate(text)
    st.subheader("Amharic translation")
    st.write(translation if translation else "(model produced an empty output — try a shorter/simpler sentence)")

with st.expander("About"):
    st.write(
        "This app uses an Attention-based Seq2Seq LSTM model trained from scratch on a "
        "parallel English-Amharic corpus. See the accompanying notebook for data preprocessing, "
        "training, evaluation (BLEU/chrF), and error analysis."
    )

### 7.3 (Optional) FastAPI alternative, matching the requested `/translate` contract

```json
POST /translate
{"text": "I am going to the university."}
->
{"translation": "ወደ ዩኒቨርሲቲ እሄዳለሁ።"}
```


In [ ]:
%%writefile api.py
from fastapi import FastAPI
from pydantic import BaseModel
from inference import Translator

app = FastAPI(title="English-Amharic NMT API")
translator = Translator(artifact_dir="deployment_artifacts")

class TranslateRequest(BaseModel):
    text: str

class TranslateResponse(BaseModel):
    translation: str

@app.post("/translate", response_model=TranslateResponse)
def translate(req: TranslateRequest):
    return {"translation": translator.translate(req.text)}

# Run with: uvicorn api:app --reload

### 7.4 Quick sanity check (within this notebook, before deploying)

In [ ]:
translation_test = model2.translate(
    torch.tensor(src_vocab.encode(tokenize_en("I am going to the university.")),
                 dtype=torch.long).unsqueeze(1).to(DEVICE),
    src_vocab, trg_vocab
)
print("Translation:", " ".join(translation_test))

---
## How to run this project end-to-end

1. Place the dataset CSV next to this notebook and set `DATA_PATH` in cell 1 (Section 1).
2. Set `QUICK_MODE = False` and choose real hyperparameters/epoch counts once the pipeline runs cleanly end-to-end in quick mode.
3. Run all cells top to bottom (Sections 2 → 6) to preprocess, train both models, evaluate, and analyze errors — fill in the actual numbers into your report tables/screenshots.
4. Section 7 saves `deployment_artifacts/` and writes `inference.py`, `app.py`, `api.py` to disk.
5. Deploy: `pip install streamlit && streamlit run app.py` (or `pip install fastapi uvicorn && uvicorn api:app --reload` for the REST API version).
6. If training is too slow on CPU, run this same notebook on Google Colab with a GPU runtime (`Runtime > Change runtime type > GPU`), or subsample the dataset further via `QUICK_MODE`.
